# Задание 2. Хаффман + восстановление кодом Хэмминга

Сначала кодируем текст Хаффманом, затем защищаем битовую последовательность кодом Хэмминга `(7,4)`. Канал вносит не более одной ошибки в выбранный 7-битный блок, поэтому Хэмминг может восстановить данные.

In [ ]:
import heapq
import itertools
import math
import random
import re
from collections import Counter
from dataclasses import dataclass

import pandas as pd
import requests

## 1. Текст

In [ ]:
BOOK_URL = "https://www.gutenberg.org/ebooks/28054.txt.utf-8"
TEXT_LIMIT = 50_000

response = requests.get(BOOK_URL, timeout=30)
response.raise_for_status()

raw_text = response.text.replace("\r\n", "\n").replace("\r", "\n")


def remove_gutenberg_wrapper(text):
    start = re.search(r"\*\*\* START OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*", text)
    end = re.search(r"\*\*\* END OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*", text)

    if start:
        text = text[start.end():]
    if end:
        text = text[:end.start()]

    return text.strip()


text = remove_gutenberg_wrapper(raw_text)[:TEXT_LIMIT]
print("Символов:", len(text))
print(text[:300])

## 2. Хаффман: блоки из 1 и 2 символов

In [ ]:
@dataclass
class Node:
    symbol: object = None
    left: object = None
    right: object = None

    def is_leaf(self):
        return self.symbol is not None


def split_blocks(sequence, block_size):
    return [sequence[i:i + block_size] for i in range(0, len(sequence), block_size)]


def build_huffman_tree(frequencies):
    order = itertools.count()
    heap = [(freq, next(order), Node(symbol=symbol)) for symbol, freq in frequencies.items()]
    heapq.heapify(heap)

    while len(heap) > 1:
        freq_1, _, node_1 = heapq.heappop(heap)
        freq_2, _, node_2 = heapq.heappop(heap)
        parent = Node(left=node_1, right=node_2)
        heapq.heappush(heap, (freq_1 + freq_2, next(order), parent))

    return heap[0][2]


def build_codes(root):
    codes = {}

    def walk(node, prefix):
        if node.is_leaf():
            codes[node.symbol] = prefix or "0"
            return

        walk(node.left, prefix + "0")
        walk(node.right, prefix + "1")

    walk(root, "")
    return codes


def encode_blocks(blocks, codes):
    return "".join(codes[block] for block in blocks)


def decode_bits(bits, root):
    result = []
    node = root

    for bit in bits:
        node = node.left if bit == "0" else node.right

        if node.is_leaf():
            result.append(node.symbol)
            node = root

    return result


def entropy(frequencies):
    total = sum(frequencies.values())
    result = 0

    for freq in frequencies.values():
        p = freq / total
        result -= p * math.log2(p)

    return result


def average_code_length(frequencies, codes):
    total = sum(frequencies.values())
    return sum((freq / total) * len(codes[symbol]) for symbol, freq in frequencies.items())


def make_huffman_result(text, block_size):
    blocks = split_blocks(text, block_size)
    frequencies = Counter(blocks)
    tree = build_huffman_tree(frequencies)
    codes = build_codes(tree)
    bits = encode_blocks(blocks, codes)
    restored = "".join(decode_bits(bits, tree))
    assert restored == text

    block_entropy = entropy(frequencies)
    block_avg_length = average_code_length(frequencies, codes)
    symbols_per_block = len(text) / len(blocks)

    return {
        "block_size": block_size,
        "tree": tree,
        "codes": codes,
        "bits": bits,
        "entropy_per_block": block_entropy,
        "avg_len_per_block": block_avg_length,
        "entropy_per_symbol": block_entropy / symbols_per_block,
        "avg_len_per_symbol": block_avg_length / symbols_per_block,
    }


huffman_1 = make_huffman_result(text, 1)
huffman_2 = make_huffman_result(text, 2)

pd.DataFrame([
    {key: value for key, value in result.items() if key not in ["tree", "codes", "bits"]}
    for result in [huffman_1, huffman_2]
])

Для демонстрации Хэмминга дальше берем поток Хаффмана для блоков из одного символа.

In [ ]:
huffman_bits = huffman_1["bits"]
huffman_tree = huffman_1["tree"]

print("Бит после Хаффмана:", len(huffman_bits))

## 3. Код Хэмминга `(7,4)`

In [ ]:
def hamming74_encode_nibble(nibble):
    d1, d2, d3, d4 = (int(bit) for bit in nibble)

    p1 = d1 ^ d2 ^ d4
    p2 = d1 ^ d3 ^ d4
    p4 = d2 ^ d3 ^ d4

    return f"{p1}{p2}{d1}{p4}{d2}{d3}{d4}"


def hamming74_encode(bit_string):
    padding = (-len(bit_string)) % 4
    padded = bit_string + "0" * padding

    blocks = [
        hamming74_encode_nibble(padded[i:i + 4])
        for i in range(0, len(padded), 4)
    ]

    return "".join(blocks), padding


def hamming74_decode_block(block):
    bits = [int(bit) for bit in block]
    p1, p2, d1, p4, d2, d3, d4 = bits

    s1 = p1 ^ d1 ^ d2 ^ d4
    s2 = p2 ^ d1 ^ d3 ^ d4
    s4 = p4 ^ d2 ^ d3 ^ d4
    syndrome = s1 + 2 * s2 + 4 * s4

    if syndrome != 0:
        bits[syndrome - 1] ^= 1

    data = f"{bits[2]}{bits[4]}{bits[5]}{bits[6]}"
    return data, syndrome


def hamming74_decode(encoded_bits, padding):
    decoded_blocks = []
    corrected_blocks = 0

    for i in range(0, len(encoded_bits), 7):
        data, syndrome = hamming74_decode_block(encoded_bits[i:i + 7])
        decoded_blocks.append(data)
        corrected_blocks += int(syndrome != 0)

    decoded = "".join(decoded_blocks)
    if padding:
        decoded = decoded[:-padding]

    return decoded, corrected_blocks

## 4. Канал и восстановление

In [ ]:
def flip_bit(bit):
    return "1" if bit == "0" else "0"


def channel_one_error_per_block(encoded_bits, block_error_probability=0.03, seed=42):
    rng = random.Random(seed)
    noisy = list(encoded_bits)
    changed_blocks = 0

    for start in range(0, len(noisy), 7):
        if rng.random() < block_error_probability:
            position = start + rng.randrange(7)
            noisy[position] = flip_bit(noisy[position])
            changed_blocks += 1

    return "".join(noisy), changed_blocks


protected_bits, padding = hamming74_encode(huffman_bits)
noisy_bits, changed_blocks = channel_one_error_per_block(protected_bits)
restored_huffman_bits, corrected_blocks = hamming74_decode(noisy_bits, padding)
restored_text = "".join(decode_bits(restored_huffman_bits, huffman_tree))

print("Блоков испорчено каналом:", changed_blocks)
print("Блоков исправлено Хэммингом:", corrected_blocks)
print("Поток Хаффмана восстановлен:", restored_huffman_bits == huffman_bits)
print("Текст восстановлен:", restored_text == text)

assert restored_huffman_bits == huffman_bits
assert restored_text == text

## Вывод

Хаффман сжимает текст, но не исправляет ошибки. Код Хэмминга `(7,4)` добавляет проверочные биты и позволяет восстановить поток, если в каждом 7-битном блоке возникла не более чем одна ошибка.